Notebook that handles a bunch of non-workforce special demand categories:

- Education (higher ed)
  - Existing-residential only; campus housing is pending data
  - prefix: `UNI`
    - (Following Depot, it covers: universities, technical colleges (polytechnics & ITE) and junior colleges; currently will only implement everything except JCs - that data is hard to come by)
- Education (all other)
  - all existing-residential
- Airports
  - Split between existing-residential and hotels
  - prefix: `AIR`
- Seaports: NIL
- Tourism
  - Split between existing-residential and hotels
  - prefixes:
    - `AMU` for amusement parks; `ZOO`; `PRK` for parks
    - There are a bunch more, see Depot repo

See `docs/non-workforce-special-demand-architecture.md`.

# Setup

In [4]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src import data_filepaths
from src.data_manual.airports import AIRPORTS
from src.data_manual.education_higher import EDU_HIGHER_ED
from src.utils.special_demand import airport_destinations, university_destinations

ALLOW_EXPORT = True


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Education (Higher Education)

Use `EDU_HIGHER_ED` from `src/data_manual/education_higher.py`: universities,
arts institutions, polytechnics, and ITE campuses. Each campus's student count
represents one modeled daily commute per student, before rounding in notebook 4.
Point IDs are `UNI_<campus.name_short>` with underscores between words.

All demand comes from existing residential origins. No hotel share or campus
housing is estimated; on-campus housing demand awaits a separate data source.
Institution and campus notes are retained for future housing integration.


In [5]:
university_demand = university_destinations(EDU_HIGHER_ED)
university_demand[["name", "institution", "daily_demand"]]


,name,institution,daily_demand
point_id,,,
UNI_NUS_West,National University of Singapore (West),NUS,12451.0
UNI_NUS_South,National University of Singapore (South),NUS,20358.0
UNI_NUS_North,National University of Singapore (North),NUS,14391.0
UNI_NUS_UTown,National University of Singapore (UTown),NUS,1316.0
UNI_NIE,National Institute of Education,NaN,2771.0
UNI_NTU_North,Nanyang Technical University (North),NTU,23703.0
UNI_NTU_South,Nanyang Technical University (South),NTU,16904.0
UNI_SMU,Singapore Management University,NaN,13049.0
UNI_SUSS,Singapore University of Social Sciences,NaN,5500.0


In [6]:
if ALLOW_EXPORT:
    university_path = Path(data_filepaths.UNIVERSITY_DEMAND)
    university_path.parent.mkdir(parents=True, exist_ok=True)
    university_demand.to_parquet(university_path)
    print(f"Exported {len(university_demand)} higher-education demand points to {university_path}")


Exported 21 higher-education demand points to src\data_intermediate\special_demand\nonworkforce\university_demand.parquet


# Airports/Seaports

Only airports for now, seaport data is quite difficult to handle.

## Airports

Annual passenger totals in `src.data_manual.airports` are modeled demand inputs.
Divide by 365 without adding a return-trip multiplier. Terminal capacities determine
within-airport shares. 

SIN's hotel share is 0.6, approximated from international visitor arrival count data.
XSP's hotel share remains a provisional 0.5. This export contains no workforce demand.

Destination IDs follow `AIR_<terminal-name>`, with underscores between words,
for example `AIR_SIN_Terminal_1` and `AIR_XSP_Terminal`.


In [7]:
HOTEL_SHARES = {"SIN": 0.6, "XSP": 0.5}
DAYS_PER_YEAR = 365

airport_demand = airport_destinations(AIRPORTS, HOTEL_SHARES, days_per_year=DAYS_PER_YEAR)
airport_demand[["name", "daily_demand", "hotel_share"]]


,name,daily_demand,hotel_share
point_id,,,
AIR_SIN_T1,SIN T1,16803.652968,0.6
AIR_SIN_T2,SIN T2,19604.261796,0.6
AIR_SIN_T3,SIN T3,15403.348554,0.6
AIR_SIN_T4,SIN T4,11202.435312,0.6
AIR_XSP,XSP,400.000000,0.5


In [8]:
if ALLOW_EXPORT:
    airport_path = Path(data_filepaths.AIRPORT_DEMAND)
    airport_path.parent.mkdir(parents=True, exist_ok=True)
    airport_demand.to_parquet(airport_path)
    print(f"Exported {len(airport_demand)} terminal demand points to {airport_path}")


Exported 5 terminal demand points to src\data_intermediate\special_demand\nonworkforce\airport_demand.parquet


# Tourism